|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Reading through the table<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: write the paged attention oracle<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn.functional as F

import cudalib
from tests.helpers import build_paged, rand_kv

Write the reference implementation of paged attention.

This is stage 07. Speed is not the point. This function is the **oracle**. You
check every later kernel against it.

It must stay correct on a shuffled pool. It must stay correct on ragged
lengths. It must stay correct when the unused slots hold another request's
tokens.

Make it correct, and accept that it is slow. Stage 08 recovers the speed.

In [2]:
### run this cell

torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

S, H, KVH, D, L = 4, 8, 2, 64, 100
BLOCK = 16

K, V = rand_kv(S, KVH, L, D, dev)
q    = torch.randn(S, H, D, device=dev)
kc, vc, block_tables, context_lens = build_paged(K, V, BLOCK)

print(f'pool {tuple(kc.shape)}, block table {tuple(block_tables.shape)}')
print(f'sequence 0 lives in blocks {block_tables[0].tolist()}')

pool (28, 2, 16, 64), block table (4, 7)
sequence 0 lives in blocks [3, 14, 10, 17, 5, 0, 7]


# Exercise 1: the dense oracle first

Use no paging. Compute attention over the original contiguous tensors. This
gives you something to check the paged version against.

In [3]:
def reference_attention(q, K, V, scale=None):
  S, H, D = q.shape
  group   = H // K.shape[1]
  scale   = scale or 1.0/math.sqrt(D)
  out = torch.empty_like(q)
  for s in range(S):
    for h in range(H):
      sc = (K[s, h//group] @ q[s,h]) * scale
      out[s,h] = torch.softmax(sc, dim=0) @ V[s, h//group]
  return out

want = reference_attention(q, K, V)
print('oracle:', tuple(want.shape))

oracle: (4, 8, 64)


# Exercise 2: the scatter

Attention cannot read the pool until something writes it. You receive one flat
slot per token. Put K and V in the correct places.

In [4]:
def write_kv(key_cache, value_cache, key, value, slot_indices):
  """key/value (T, KVH, D), slot_indices (T,) flat slots."""
  BS = key_cache.shape[2]
  for i, slot in enumerate(slot_indices.tolist()):
    b, off = slot // BS, slot % BS
    key_cache[b, :, off]   = key[i]
    value_cache[b, :, off] = value[i]

kc2 = torch.zeros_like(kc); vc2 = torch.zeros_like(vc)
slots = torch.tensor([block_tables[0,0]*BLOCK + 0, block_tables[0,0]*BLOCK + 1])
write_kv(kc2, vc2, K[0,:, :2].permute(1,0,2), V[0,:, :2].permute(1,0,2), slots)
print('wrote 2 tokens; matches source:',
      torch.allclose(kc2[block_tables[0,0], :, :2], K[0,:, :2]))

wrote 2 tokens; matches source: True


# Exercise 3: the gather

Walk the block table. Collect the blocks. Cut the result to `context_len`.
Then do the arithmetic from Exercise 1.

In [5]:
def paged_attention(q, kc, vc, block_tables, context_lens, scale=None):
  S, H, D = q.shape
  KVH, BS = kc.shape[1], kc.shape[2]
  group   = H // KVH
  scale   = scale or 1.0/math.sqrt(D)
  out = torch.empty_like(q)

  for s in range(S):
    n      = int(context_lens[s])
    blocks = block_tables[s, :(n + BS - 1)//BS].long()
    k = kc[blocks].permute(1,0,2,3).reshape(KVH, -1, D)[:, :n]
    v = vc[blocks].permute(1,0,2,3).reshape(KVH, -1, D)[:, :n]
    for h in range(H):
      kvh = h // group
      sc  = (k[kvh] @ q[s,h]) * scale
      out[s,h] = torch.softmax(sc, dim=0) @ v[kvh]
  return out

got = paged_attention(q, kc, vc, block_tables, context_lens)
print('max difference from the oracle:', (got - want).abs().max().item())

max difference from the oracle: 0.0


# Exercise 4: two quiet failures

The allocator recycles blocks, so the slots past `context_len` hold another
request's tokens. And no two sequences have the same length.

In [6]:
kc3, vc3 = kc.clone(), vc.clone()
for s in range(S):
  for b in range(block_tables.shape[1]):
    phys = int(block_tables[s,b])
    for off in range(BLOCK):
      if b*BLOCK + off >= L:
        kc3[phys,:,off] = 999.0
        vc3[phys,:,off] = 999.0

after = paged_attention(q, kc3, vc3, block_tables, context_lens)
print('output unchanged:', torch.allclose(got, after, atol=1e-5))

ragged = torch.tensor([100, 1, 17, 64], dtype=torch.int32, device=dev)
r = paged_attention(q, kc, vc, block_tables, ragged)
print('ragged context lengths ran:', tuple(r.shape))

output unchanged: True
ragged context lengths ran: (4, 8, 64)


# Exercise 5: what did paging cost?

In [7]:
if dev == 'cuda':
  S, H, KVH, D, L = 32, 16, 8, 128, 512
  K2, V2 = rand_kv(S, KVH, L, D, dev, dtype=torch.float16)
  q2 = torch.randn(S, H, D, device=dev, dtype=torch.float16)
  kcb, vcb, btb, ctxb = build_paged(K2, V2, BLOCK)

  paged = cudalib.bench_ms(lambda: paged_attention(q2, kcb, vcb, btb, ctxb),
                           iters=5, warmup=2)
  dense = cudalib.bench_ms(lambda: F.scaled_dot_product_attention(
            q2.unsqueeze(2), K2.repeat_interleave(H//KVH,1),
            V2.repeat_interleave(H//KVH,1)), iters=20, warmup=5)
  print(f'contiguous SDPA: {dense:8.3f} ms')
  print(f'your paged loop: {paged:8.3f} ms   ({paged/dense:.0f}x slower)')

contiguous SDPA:    0.886 ms
your paged loop:   13.445 ms   (15x slower)


### You built the oracle, and you received the bill

Your `paged_attention` is correct on a shuffled pool, on ragged lengths, and
with recycled blocks. Keep it. Every kernel in the next three stages uses this
function as its only reference.

It also runs about fifteen times slower than the contiguous version. You wrote
both reasons yourself:

- **A Python loop over sequences.** One gather and one matmul per sequence per
  head. That is thousands of kernel launches where one belongs.
- **`kc[blocks]` materialises the data.** You allocate the whole gathered K
  and V, write them to memory, and read them back to compute a softmax. At 512
  tokens of context that is more traffic than the cache itself.

One kernel removes both problems. The second fix is the interesting one. The
score row never has to exist. Fold each tile into a running softmax instead.

    ./vc guide 8